# Load an open-source circuit-tracer CLT in CLT-Forge

This notebook loads a trained circuit-tracer CLT from HuggingFace, runs attribution through CLT-Forge's attribution runner, saves a CLT-Forge-compatible graph artifact, optionally converts circuit-tracer feature metadata into CLT-Forge feature JSON files, and opens the existing CLT-Forge visual interface.

It does not change the visual interface. The bridge happens in the Python library layer.

In [ ]:
from pathlib import Path

import torch

from clt_forge.attribution.attribution import AttributionRunner
from clt_forge.attribution.circuit_tracer_features import (
    download_clt_forge_feature_dicts_for_graph,
)
from clt_forge.frontend.app import main
from clt_forge.frontend.config.settings import AppConfig

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if device == "cuda" else torch.float32

# circuit-tracer open-source CLT refs listed in the vendored circuit-tracer README:
# - mntss/clt-gemma-2-2b-426k
# - mntss/clt-gemma-2-2b-2.5M
# - mntss/clt-llama-3.2-1b-524k
model_name = "google/gemma-2-2b"
circuit_tracer_clt = "mntss/clt-gemma-2-2b-426k"

output_dir = Path("outputs/circuit_tracer_gemma_demo")
graph_path = output_dir / "attribution_graph.pt"
feature_dict_dir = output_dir / "feature_dicts"
output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
runner = AttributionRunner.from_circuit_tracer_hub(
    hf_ref=circuit_tracer_clt,
    model_name=model_name,
    device=device,
    dtype=dtype,
    backend="transformerlens",
    lazy_encoder=False,
    lazy_decoder=True,
    debug=False,
)

result = runner.run(
    input_string="The capital of France is",
    folder_name=str(output_dir),
    graph_name=graph_path.name,
    max_n_logits=5,
    max_feature_nodes=4096,
    batch_size=128,
    offload="cpu",
    run_interventions=False,
)

graph_path

In [ ]:
# Optional: pull circuit-tracer feature examples for the graph's active features
# and convert them to the CLT-Forge frontend feature JSON layout.
# Start with a small max_features while exploring; downloading every active
# feature can be slow for large graphs.
written_feature_files = download_clt_forge_feature_dicts_for_graph(
    graph_result=result,
    scan=result.get("circuit_tracer_scan", circuit_tracer_clt),
    output_dir=feature_dict_dir,
    max_features=50,
    strict=False,
)

len(written_feature_files)

In [ ]:
cfg = AppConfig(
    attr_graph_path=str(graph_path),
    dict_base_folder=str(feature_dict_dir),
    clt_checkpoint="",
    model_name=model_name,
    model_class_name="HookedTransformer",
    port=8106,
)

main(cfg)